# IBKR FINAL GATE — NQ / ES / TSM / SOXX

Run the pilot once to verify OAuth and paths. Then set `PILOT = False` and run all cells once for the publishable result. A pilot always returns `PILOT_ONLY` and is never a backtest baseline. Raw Drive data is read only.

In [ ]:
# Install a clean copy so an old Colab module cache cannot change gate semantics.
!rm -rf /content/IBKR-market-data-cleaning-pipeline
!git clone --depth 1 https://github.com/hh4832/IBKR-market-data-cleaning-pipeline.git /content/IBKR-market-data-cleaning-pipeline
!python -m pip install -q -e '/content/IBKR-market-data-cleaning-pipeline[colab]'

import json, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

for name in list(sys.modules):
    if name == 'ibkr_qc' or name.startswith('ibkr_qc.'):
        del sys.modules[name]

repo = Path('/content/IBKR-market-data-cleaning-pipeline')
print('Git SHA:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from ibkr_qc.drive_io import authorized_drive
from ibkr_qc.final_gate import DEFAULT_IDS, FinalGateConfig, run_final_gate

# First run: True. Publishable one-shot run: change to False.
PILOT = True

# Existing source IDs confirmed in Drive. OUTPUT is 00_Quant_Research_Hub.
FOLDER_IDS = {
    **DEFAULT_IDS,
    'OUTPUT': '1CIlmaQv6BsAnMlNHxDTanJB4JttqYbs9',
}
CACHE_ROOT = Path('/content/drive/MyDrive/00Quant_Research/00_Cache/IBKR_FINAL_GATE_V030')
OUTPUT_ROOT = Path('/content/ibkr_final_gate_outputs')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config = FinalGateConfig(
    folder_ids=FOLDER_IDS,
    upload=True,
    max_equity_chunks=5 if PILOT else 0,
    max_futures_contracts=1 if PILOT else 0,
)
service = authorized_drive()
run_dir = run_final_gate(service, config, cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT)
print('Local run:', run_dir)
print((run_dir / 'FINAL_VERDICT.txt').read_text().strip())
if PILOT:
    assert (run_dir / 'FINAL_VERDICT.txt').read_text().strip() == 'PILOT_ONLY'

In [ ]:
# Compact readback. Copy this output to the Notion handoff after the FULL run.
manifest = json.loads((run_dir / 'final_gate_manifest.json').read_text())
print('Overall:', manifest['overall_verdict'])
print('Instrument verdicts:', manifest['instrument_verdicts'])
print('Git SHA:', manifest['git_sha'])
for result in manifest['results']:
    print(result['symbol'], result['verdict'], 'errors=', len(result.get('errors', [])),
          'unresolved=', result.get('unresolved_minutes', 0))